# 🎓 AI Assignment Grader
**Output:** Total score / 25 · Strengths · Areas to Improve (with specific mistakes, why it matters, fix)

### How to use
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. Wait for the Gradio UI at the bottom, then upload your files

---

## Cell 1 — Install Everything

In [ ]:
import subprocess, sys

# ── System packages (zstd required by Ollama install script) ──────────────────
print('📦 Installing system packages...')
subprocess.run(
    ['apt-get', 'install', '-y', '-q', 'zstd', 'pciutils', 'lshw'],
    check=True, capture_output=True
)
print('✅ System packages ready')

# ── Ollama ────────────────────────────────────────────────────────────────────
print('\n📦 Installing Ollama (streaming output)...')
proc = subprocess.Popen(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    line = line.rstrip()
    if line: print(f'  {line}')
proc.wait()

binary = subprocess.run(['which', 'ollama'], capture_output=True, text=True)
if binary.returncode != 0:
    raise RuntimeError('Ollama binary not found — see output above')
ver = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
print(f'\n✅ Ollama installed: {ver.stdout.strip()}')

# ── Python packages ───────────────────────────────────────────────────────────
print('\n📦 Installing Python packages...')
pkgs = ['gradio>=4.0.0', 'PyMuPDF', 'nbformat', 'requests', 'tenacity']
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    print(f"  {'✅' if r.returncode==0 else '❌'} {pkg}")

print('\n🎉 All installations complete!')

## Cell 2 — Start Ollama & Pull Model

In [ ]:
import subprocess, time, os, requests

# ── Change this to 'qwen2.5-coder:3b' for faster CPU inference ───────────────
MODEL_NAME = 'qwen2.5-coder:7b'

# ── Environment vars ──────────────────────────────────────────────────────────
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'
os.environ['OLLAMA_HOST']     = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS']  = '*'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'   # keep model in RAM permanently

# ── Start server ──────────────────────────────────────────────────────────────
print('🚀 Starting Ollama server...')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'], env=os.environ.copy(),
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f'   PID: {ollama_proc.pid}')

print('⏳ Waiting for server', end='')
for i in range(40):
    try:
        if requests.get('http://localhost:11434', timeout=2).status_code == 200:
            print(f' ready ({i+1}s) ✅'); break
    except:
        print('.', end='', flush=True); time.sleep(1)
else:
    raise RuntimeError('Ollama server failed to start')

# ── Pull model ────────────────────────────────────────────────────────────────
print(f'\n📥 Pulling {MODEL_NAME} (~4.5 GB on first run)...')
pull = subprocess.Popen(
    ['ollama', 'pull', MODEL_NAME],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
last = ''
for line in pull.stdout:
    line = line.rstrip()
    if line and line != last:
        print(f'  {line}'); last = line
pull.wait()
if pull.returncode != 0:
    raise RuntimeError('Model pull failed')

# ── Warm up (eliminates cold-start delay for first user) ─────────────────────
print('\n🔥 Warming up model...')
requests.post(
    'http://localhost:11434/api/generate',
    json={'model': MODEL_NAME, 'prompt': '', 'keep_alive': -1},
    timeout=60
)
print('✅ Model warm and ready!')
print(f'\n🎉 {MODEL_NAME} loaded — proceed to Cell 3')

## Cell 3 — Grading Logic

In [ ]:
import json, re, textwrap, subprocess, tempfile, os, traceback
import requests
import fitz       # PyMuPDF
import nbformat
from tenacity import retry, stop_after_attempt, wait_fixed

MODEL_NAME   = 'qwen2.5-coder:7b'   # must match Cell 2
OLLAMA_URL   = 'http://localhost:11434/api/chat'
MAX_CODE_LEN = 3500   # chars — keeps prompt tight for faster inference
TIMEOUT      = 300

# ─────────────────────────────────────────────────────────────────────────────
# FILE PARSERS
# ─────────────────────────────────────────────────────────────────────────────

def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF. Raises if empty/image-only."""
    doc = fitz.open(pdf_path)
    pages = [page.get_text('text').strip() for page in doc]
    doc.close()
    full = '\n\n'.join(p for p in pages if p)
    if not full.strip():
        raise ValueError('PDF appears empty or image-only (no extractable text).')
    # Trim to key info — first 1500 chars contains task description
    return full[:1500] if len(full) > 1500 else full


def extract_notebook_code(ipynb_path: str) -> str:
    """Extract code + markdown cells from .ipynb with separators."""
    nb = nbformat.read(ipynb_path, as_version=4)
    parts = []
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'code':
            src = cell.source.strip()
            if src:
                parts.append(f'# ── Code Cell {i+1} ──\n{src}')
        elif cell.cell_type == 'markdown':
            src = cell.source.strip()
            if src:
                parts.append(
                    f'# ── Markdown Cell {i+1} ──\n# '
                    + '\n# '.join(src.splitlines())
                )
    if not parts:
        raise ValueError('Notebook has no code cells.')
    code = '\n\n'.join(parts)
    if len(code) > MAX_CODE_LEN:
        code = code[:MAX_CODE_LEN] + f'\n\n# ... [truncated to {MAX_CODE_LEN} chars]'
    return code


def run_code_sandbox(ipynb_path: str, timeout: int = 30) -> dict:
    """Execute notebook code in a subprocess sandbox. Returns result dict."""
    try:
        nb = nbformat.read(ipynb_path, as_version=4)
        code_lines = []
        for cell in nb.cells:
            if cell.cell_type == 'code':
                lines = [
                    l for l in cell.source.splitlines()
                    if not l.strip().startswith(('!', '%'))
                ]
                if lines:
                    code_lines.extend(lines)
                    code_lines.append('')
        with tempfile.NamedTemporaryFile(
            suffix='.py', delete=False, mode='w'
        ) as f:
            f.write('\n'.join(code_lines))
            tmp_py = f.name
        result = subprocess.run(
            ['python', tmp_py],
            capture_output=True, text=True, timeout=timeout
        )
        os.unlink(tmp_py)
        return {
            'success': result.returncode == 0,
            'stdout': result.stdout[:2000],
            'stderr': result.stderr[:1000],
            'error': None,
        }
    except subprocess.TimeoutExpired:
        return {'success': False, 'stdout': '', 'stderr': '',
                'error': f'Timed out after {timeout}s'}
    except Exception as e:
        return {'success': False, 'stdout': '', 'stderr': '', 'error': str(e)}


# ─────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPT  — strict, forensic, structured output
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = textwrap.dedent('''
    You are a strict but fair programming instructor grading a student Jupyter notebook.

    You will receive:
    - An assignment question
    - A grading rubric with criteria, point values, and explicit penalty rules
    - The student code extracted from their notebook
    - Optionally: sandbox execution output

    Your job:
    1. Internally evaluate each rubric criterion and sum the scores to get total_score out of 25.
    2. Identify genuine strengths — be specific, cite actual code or techniques used by the student.
    3. Identify critical areas to improve. This is the most important section. Be forensic and precise.
       For each issue you find, you MUST provide:
         - category: one of [Bug, Code Quality, Data Preprocessing, Modeling, Missing Requirement]
         - issue: the exact mistake — name the variable, function, or line if possible
         - why_it_matters: the consequence — wrong result, data leakage, misleading output, etc.
         - fix: a concrete correction, ideally with a short corrected code snippet

       Categories to check:
         • Bug — wrong logic, incorrect calculation, crashes
         • Code Quality — no docstrings, magic numbers, poor variable names, repeated code
         • Data Preprocessing — wrong imputation strategy, no scaling, encoding errors, leakage
         • Modeling — wrong metric, splitting after scaling (leakage), no cross-validation
         • Missing Requirement — something explicitly required by the rubric that is absent

    STRICT RULES:
    - Do NOT be vague. "Improve variable names" is rejected.
      Write: "Variable df2 in cell 3 should be named df_cleaned to reflect its purpose."
    - Do NOT invent mistakes. Only report what you can see in the code.
    - If execution output is provided, use it as evidence (e.g., exceptions prove a bug).
    - Respond ONLY with valid JSON. No markdown, no text outside the JSON object.

    Return exactly this schema — nothing else:
    {
      "total_score": <integer out of 25>,
      "strengths": [
        "Specific strength with reference to actual code or technique used"
      ],
      "areas_to_improve": [
        {
          "category": "<Bug | Code Quality | Data Preprocessing | Modeling | Missing Requirement>",
          "issue": "Precise description naming the specific mistake",
          "why_it_matters": "The consequence or reason this is wrong",
          "fix": "Concrete suggestion or corrected code snippet"
        }
      ]
    }
''').strip()


# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_user_prompt(
    question: str, rubric: str, code: str, execution: dict | None
) -> str:
    parts = [
        f'=== ASSIGNMENT QUESTION ===\n{question.strip()}',
        f'=== GRADING RUBRIC ===\n{rubric.strip()}',
        f'=== STUDENT CODE ===\n{code.strip()}',
    ]
    if execution:
        status = '✓ ran successfully' if execution['success'] else '✗ failed'
        block  = f'Status: {status}\n'
        if execution['stdout']: block += f"Output:\n{execution['stdout']}\n"
        if execution['stderr']: block += f"Errors:\n{execution['stderr']}\n"
        if execution['error']:  block += f"Exception: {execution['error']}\n"
        parts.append(f'=== EXECUTION RESULTS ===\n{block}')
    parts.append(
        '=== YOUR TASK ===\n'
        'Evaluate the student submission and return ONLY the JSON schema specified.'
    )
    return '\n\n'.join(parts)


# ─────────────────────────────────────────────────────────────────────────────
# OLLAMA CALL  — retries 3× on failure
# ─────────────────────────────────────────────────────────────────────────────

@retry(stop=stop_after_attempt(3), wait=wait_fixed(5))
def call_ollama(
    question: str, rubric: str, code: str, execution: dict | None
) -> dict:
    payload = {
        'model': MODEL_NAME,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_user_prompt(question, rubric, code, execution)},
        ],
        'stream': False,
        'format': 'json',
        'options': {
            'temperature': 0.1,
            'num_predict': 1500,
            'num_ctx':     4096,
            'num_thread':  4,
            'num_batch':   512,
        },
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
    resp.raise_for_status()
    raw = resp.json()['message']['content']
    # Strip accidental markdown fences
    raw = re.sub(r'^```json\s*', '', raw.strip())
    raw = re.sub(r'```$',        '', raw.strip())
    return json.loads(raw)


# ─────────────────────────────────────────────────────────────────────────────
# HTML REPORT RENDERER
# ─────────────────────────────────────────────────────────────────────────────

def grade_to_color(pct: float) -> str:
    if pct >= 0.85: return '#22c55e'
    if pct >= 0.70: return '#84cc16'
    if pct >= 0.55: return '#f59e0b'
    if pct >= 0.40: return '#f97316'
    return '#ef4444'


CATEGORY_COLORS = {
    'Bug':                 ('#fee2e2', '#dc2626', '#fca5a5'),
    'Code Quality':        ('#fef9c3', '#ca8a04', '#fde047'),
    'Data Preprocessing':  ('#ede9fe', '#7c3aed', '#c4b5fd'),
    'Modeling':            ('#fff7ed', '#ea580c', '#fdba74'),
    'Missing Requirement': ('#f0f9ff', '#0284c7', '#7dd3fc'),
}


def render_html_report(result: dict) -> str:
    total = result.get('total_score', 0)
    pct   = total / 25
    color = grade_to_color(pct)

    if   pct >= 0.90: grade = 'A+'
    elif pct >= 0.80: grade = 'A'
    elif pct >= 0.70: grade = 'B'
    elif pct >= 0.60: grade = 'C'
    elif pct >= 0.50: grade = 'D'
    else:             grade = 'F'

    # ── Strengths ──────────────────────────────────────────────────────────────
    strengths_li = ''.join(
        f"<li style='margin-bottom:8px;line-height:1.6'>{s}</li>"
        for s in result.get('strengths', [])
    )

    # ── Areas to improve cards ─────────────────────────────────────────────────
    improve_cards = ''
    for item in result.get('areas_to_improve', []):
        cat   = item.get('category', 'Code Quality')
        bg, text, border = CATEGORY_COLORS.get(cat, ('#f8fafc', '#475569', '#cbd5e1'))
        issue = item.get('issue', '')
        why   = item.get('why_it_matters', '')
        fix   = item.get('fix', '')

        # Render fix as code block if it looks like code
        is_code = any(tok in fix for tok in
                      ['\n', 'def ', 'df.', 'import ', ' = ', '()', '[]', ':'])
        fix_html = (
            f"<pre style='background:#1e293b;color:#e2e8f0;padding:10px 14px;"
            f"border-radius:6px;font-size:12px;overflow-x:auto;"
            f"margin:8px 0 0;white-space:pre-wrap'>{fix}</pre>"
            if is_code else
            f"<p style='margin:6px 0 0;font-size:13px;color:#374151'>"
            f"<b>Fix:</b> {fix}</p>"
        )

        improve_cards += f'''
        <div style="border:1px solid {border};background:{bg};border-radius:10px;
                    padding:16px;margin-bottom:14px">
          <div style="display:flex;align-items:flex-start;gap:8px;margin-bottom:8px">
            <span style="background:{text};color:white;font-size:11px;font-weight:700;
                         padding:3px 9px;border-radius:99px;white-space:nowrap;margin-top:1px"
            >{cat}</span>
            <span style="font-weight:600;font-size:14px;color:#111827;line-height:1.4"
            >{issue}</span>
          </div>
          <p style="margin:0 0 4px;font-size:13px;color:#4b5563;line-height:1.5">
            <b>Why it matters:</b> {why}
          </p>
          {fix_html}
        </div>'''

    if not improve_cards:
        improve_cards = "<p style='color:#6b7280;font-size:13px'>No major issues identified.</p>"

    n_issues = len(result.get('areas_to_improve', []))

    return f'''
    <div style="font-family:'Segoe UI',system-ui,sans-serif;max-width:780px;margin:0 auto">

      <!-- Score header -->
      <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                  border-radius:16px;padding:28px 32px;margin-bottom:20px;
                  display:flex;justify-content:space-between;align-items:center">
        <div>
          <div style="color:#94a3b8;font-size:12px;text-transform:uppercase;
                      letter-spacing:.1em;margin-bottom:4px">Total Score</div>
          <div style="color:#f1f5f9;font-size:42px;font-weight:900;line-height:1">
            {total}
            <span style="font-size:20px;font-weight:400;color:#94a3b8">/ 25</span>
          </div>
          <div style="margin-top:14px;background:#1e3a5f;border-radius:99px;
                      height:10px;width:260px">
            <div style="background:{color};width:{int(pct*100)}%;
                        height:10px;border-radius:99px"></div>
          </div>
          <div style="color:#94a3b8;font-size:13px;margin-top:6px">
            {int(pct*100)}% &middot; Qwen2.5-Coder via Ollama
          </div>
        </div>
        <div style="text-align:center">
          <div style="width:80px;height:80px;border-radius:50%;background:{color};
                      display:flex;align-items:center;justify-content:center;
                      font-size:32px;font-weight:900;color:white">{grade}</div>
          <div style="color:#94a3b8;font-size:11px;margin-top:6px">Grade</div>
        </div>
      </div>

      <!-- Strengths -->
      <div style="background:#f0fdf4;border:1px solid #bbf7d0;
                  border-radius:12px;padding:20px;margin-bottom:16px">
        <h3 style="margin:0 0 12px;color:#15803d;font-size:15px">✅ Strengths</h3>
        <ul style="margin:0;padding-left:20px;color:#166534;font-size:13px;line-height:1.7">
          {strengths_li or "<li>No specific strengths identified.</li>"}
        </ul>
      </div>

      <!-- Areas to improve -->
      <div style="background:white;border:1px solid #e5e7eb;border-radius:12px;
                  padding:20px;box-shadow:0 1px 3px rgba(0,0,0,.06)">
        <h3 style="margin:0 0 14px;font-size:15px;color:#111827">
          &#128161; Areas to Improve
          <span style="font-size:12px;font-weight:400;color:#6b7280;margin-left:6px">
            ({n_issues} issue{'s' if n_issues != 1 else ''} found)
          </span>
        </h3>
        {improve_cards}
      </div>

      <div style="text-align:center;padding:16px;color:#9ca3af;
                  font-size:11px;margin-top:8px">
        AI Assignment Grader &middot; Qwen2.5-Coder via Ollama
      </div>
    </div>'''


# ─────────────────────────────────────────────────────────────────────────────
# STATUS HELPER
# ─────────────────────────────────────────────────────────────────────────────

def _s(icon: str, label: str, msg: str) -> str:
    return (
        f"<div style='padding:8px 12px;margin-bottom:6px;background:#f8fafc;"
        f"border-radius:8px;border-left:3px solid #94a3b8;font-size:13px'>"
        f"<b>{icon} {label}:</b> {msg}</div>"
    )


# ─────────────────────────────────────────────────────────────────────────────
# MAIN GRADE FUNCTION  — generator so Gradio streams status updates live
# ─────────────────────────────────────────────────────────────────────────────

def grade_assignment(question_pdf, rubric_txt, notebook_ipynb, run_code):
    try:
        # ── Validate inputs ───────────────────────────────────────────────────
        if question_pdf is None:
            yield None, "<p style='color:red'>❌ Please upload the assignment PDF.</p>"
            return
        if rubric_txt is None:
            yield None, "<p style='color:red'>❌ Please upload the rubric TXT file.</p>"
            return
        if notebook_ipynb is None:
            yield None, "<p style='color:red'>❌ Please upload the student notebook (.ipynb).</p>"
            return

        # Normalise — Gradio can pass str path or file object
        pdf_path    = question_pdf    if isinstance(question_pdf,    str) else question_pdf.name
        rubric_path = rubric_txt      if isinstance(rubric_txt,      str) else rubric_txt.name
        nb_path     = notebook_ipynb  if isinstance(notebook_ipynb,  str) else notebook_ipynb.name

        # ── Step 1: Parse files ───────────────────────────────────────────────
        yield None, _s('⏳', 'Step 1/3', 'Parsing uploaded files...')

        question = extract_pdf_text(pdf_path)

        with open(rubric_path, 'r', encoding='utf-8') as f:
            rubric = f.read()
        if not rubric.strip():
            yield None, "<p style='color:red'>❌ Rubric file is empty.</p>"
            return

        code = extract_notebook_code(nb_path)

        step1 = _s(
            '✅', 'Step 1/3 complete',
            f'PDF: {len(question):,} chars &middot; '
            f'Rubric: {len(rubric):,} chars &middot; '
            f'Code: {len(code):,} chars'
        )
        yield None, step1

        # ── Step 2: Sandbox execution (optional) ──────────────────────────────
        execution = None
        if run_code:
            yield None, step1 + _s('⏳', 'Step 2/3', 'Running notebook in sandbox (30s timeout)...')
            execution = run_code_sandbox(nb_path, timeout=30)
            icon = '✅' if execution['success'] else '⚠️'
            msg  = 'Ran successfully' if execution['success'] \
                   else (execution.get('error') or execution.get('stderr', ''))[:120]
            step2 = _s(icon, 'Step 2/3 complete', f'Sandbox: {msg}')
        else:
            step2 = _s('⏭️', 'Step 2/3', 'Code execution skipped.')
        yield None, step1 + step2

        # ── Step 3: LLM evaluation ────────────────────────────────────────────
        yield None, step1 + step2 + _s(
            '⏳', 'Step 3/3',
            'Sending to Qwen2.5-Coder for evaluation&#8230; (30&ndash;90 seconds)'
        )

        result   = call_ollama(question, rubric, code, execution)
        html     = render_html_report(result)
        json_out = json.dumps(result, indent=2)
        yield json_out, html

    except Exception as e:
        tb = traceback.format_exc()
        yield None, f"""
        <div style='background:#fee2e2;padding:16px;border-radius:10px;
                    border:1px solid #fca5a5'>
          <b style='color:#991b1b'>&#10060; Error: {e}</b>
          <pre style='font-size:11px;margin-top:8px;color:#7f1d1d;
                      overflow:auto'>{tb}</pre>
        </div>"""


print('✅ Grading logic loaded — proceed to Cell 4')

## Cell 4 — Launch Gradio UI

In [ ]:
import gradio as gr

with gr.Blocks(
    title='AI Assignment Grader',
    theme=gr.themes.Soft(primary_hue='slate'),
    css='''
        .upload-box { border: 2px dashed #cbd5e1 !important;
                      border-radius: 10px !important; }
        .grade-btn  { background: linear-gradient(135deg,#1e293b,#334155) !important;
                      color: white !important; font-weight: 700 !important;
                      font-size: 16px !important; height: 52px !important; }
    '''
) as demo:

    gr.HTML('''
    <div style="text-align:center;padding:24px 0 8px">
      <h1 style="font-size:28px;font-weight:800;margin:0;color:#0f172a">
        &#127891; AI Assignment Grader
      </h1>
      <p style="color:#64748b;margin:6px 0 0;font-size:14px">
        Powered by Qwen2.5-Coder via Ollama
        &middot; Score &middot; Strengths &middot; Detailed Improvement Feedback
      </p>
    </div>
    ''')

    with gr.Row():

        # ── Left: inputs ──────────────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('### &#128194; Upload Files')

            question_pdf = gr.File(
                label='&#128196; Assignment Question (PDF)',
                file_types=['.pdf'],
                elem_classes=['upload-box']
            )
            rubric_txt = gr.File(
                label='&#128203; Grading Rubric (TXT)',
                file_types=['.txt'],
                elem_classes=['upload-box']
            )
            notebook_ipynb = gr.File(
                label='&#128211; Student Notebook (IPYNB)',
                file_types=['.ipynb'],
                elem_classes=['upload-box']
            )
            run_code = gr.Checkbox(
                label='&#9881;&#65039; Run code in sandbox (30s timeout)',
                value=False,
                info='Executes the notebook and feeds output to the LLM for evidence-based grading.'
            )
            grade_btn = gr.Button(
                '&#128640; Grade Assignment',
                variant='primary',
                elem_classes=['grade-btn']
            )

            gr.Markdown('''
            ---
            **Output includes:**
            - Total score out of 25
            - Specific strengths with code references
            - Detailed issues: category · exact mistake · why it matters · fix

            **Tips**
            - PDF must have selectable text (not scanned image)
            - Rubric TXT: use explicit penalty rules per criterion for best results
            - Grading takes 30–90 seconds
            ''')

        # ── Right: outputs ────────────────────────────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown('### &#128202; Results')

            with gr.Tabs():
                with gr.TabItem('&#128203; Feedback Report'):
                    report_html = gr.HTML(
                        value="""
                        <div style='color:#94a3b8;padding:60px;text-align:center;
                                    font-size:15px;border:2px dashed #e2e8f0;
                                    border-radius:12px;margin:8px'>
                          Upload your files and click
                          <b>&#128640; Grade Assignment</b> to begin.<br>
                          <span style='font-size:12px;margin-top:8px;display:block'>
                            Results show: total score / 25 &middot;
                            strengths &middot; detailed improvement areas
                          </span>
                        </div>"""
                    )
                with gr.TabItem('&#128296; Raw JSON'):
                    json_output = gr.Code(
                        language='json',
                        label='Raw grading output',
                        lines=30
                    )

    grade_btn.click(
        fn=grade_assignment,
        inputs=[question_pdf, rubric_txt, notebook_ipynb, run_code],
        outputs=[json_output, report_html],
    )

demo.launch(
    share=True,
    debug=False,
    show_error=True,
    quiet=True
)

---
## Rubric Template

Copy this into your `rubric.txt` file. The more specific the penalty rules, the more precise the feedback.

```
ASSIGNMENT RUBRIC — [Assignment Title]
Total: 25 points

── CRITERION 1: Data Loading & Exploration  [5 pts] ──
Full marks: Loads dataset correctly, displays shape, dtypes, null counts, summary stats.
Penalise:
  - No .info() or .describe() called (-2)
  - Shape not checked (-1)
  - Null counts not inspected (-2)

── CRITERION 2: Data Cleaning  [7 pts] ──
Full marks: Handles nulls with appropriate strategy (median for skewed, mean for normal),
  removes duplicates, fixes dtypes, no leakage.
Penalise:
  - fillna(0) used on continuous columns (-3) — use median/mean
  - Duplicates not removed (-2)
  - Cleaning done AFTER train/test split — data leakage (-3)
  - Dtype issues not fixed (-1)

── CRITERION 3: Analysis & Visualisation  [8 pts] ──
Full marks: At least 2 meaningful plots with titles/labels, correct statistical
  interpretation, correlation analysis present.
Penalise:
  - Plots have no axis labels or title (-2)
  - No written statistical interpretation (-3)
  - Only one visualisation (-2)
  - Correlation matrix absent or misread (-2)

── CRITERION 4: Code Quality  [5 pts] ──
Full marks: PEP8 compliant, functions used, docstrings present,
  no magic numbers, meaningful variable names throughout.
Penalise:
  - No functions defined, all code in global scope (-2)
  - No comments or docstrings (-2)
  - Magic numbers hardcoded without explanation (-1)
  - Variable names like x, df2, tmp used (-1)
```